# Choosing a method: which levers, and when

`tsam.aggregate(data, n_clusters=6, period_duration="1D")` works, and it quietly makes four
decisions on your behalf. This tutorial makes those decisions explicit, so you can tell which
ones matter for *your* model and which you can leave alone.

By the end you will be able to:

1. **name** the four levers and say what each one throws away;
2. **choose** a clustering method and a representation independently, rather than accepting the
   pairing a method defaults to;
3. **decide** between the three different ways to preserve a peak — and know why they are not
   interchangeable;
4. **read** a configuration off what you are modelling.

This is the *decision* companion to
[Comparing clustering methods](comparing_clustering_methods.ipynb), which explains in depth *why*
the six clustering methods disagree. Here we take that as read and widen the question to the whole
configuration.

> We reuse that tutorial's **12 designed days** so the effects stay legible: 5 sunny, 4 cloudy,
> 2 high-load, and 1 **storm** — a single outlier holding the maximum load. `preserve_column_means=False`
> throughout, so every number below is the raw effect of the lever, unadjusted.

In [ ]:
import pandas as pd
import plotly.io as pio

import tsam
from tsam import ClusterConfig, ExtremeConfig, SegmentConfig

pio.renderers.default = "notebook_connected"

days = pd.read_csv("../data/comparison_days.csv", index_col=0, parse_dates=True)
COMMON = {"n_clusters": 3, "period_duration": "1D", "preserve_column_means": False}

print(
    f"{days.shape[0]} timesteps = 12 days x 4 six-hourly steps, attributes: {list(days.columns)}"
)
print(f"the storm's peak load: {days['load'].max()} (normalised)")

## 1  The four levers

Aggregation is not one choice, it is four — and they are independent. In pipeline order:

| Lever | Parameter | The question it answers | What it costs |
|---|---|---|---|
| **1. Grouping** | `ClusterConfig(method=…)` | which days belong together? | — |
| **2. Representation** | `ClusterConfig(representation=…)` | what does each group become? | — |
| **3. Extremes** | `extremes=ExtremeConfig(…)` | must a specific day survive? | usually +1 typical period |
| **4. Segmentation** | `segments=SegmentConfig(…)` | how much detail *within* a day? | detail inside the period |

Only levers 3 and 4 change the *size* of the problem. Levers 1 and 2 are free: they change **what
you keep** at the same size. That makes them the first place to look, and the easiest to overlook.

### The default, spelled out

A bare `aggregate()` call chooses `hierarchical` grouping with the `medoid` representation, no
extremes and no segmentation. It is a good default — deterministic, robust, and its
representatives are real days. Here is what it does to our 12 days:

In [ ]:
baseline = tsam.aggregate(days, **COMMON, cluster=ClusterConfig(method="hierarchical"))

print(f"typical periods:   {baseline.n_clusters}")
print(f"weighted RMSE:     {baseline.accuracy.weighted_rmse:.4f}")
print(
    f"peak load kept:    {baseline.cluster_representatives['load'].max():.2f}  "
    f"(the storm was {days['load'].max():.2f})"
)

**The default lost a quarter of the peak.** Nothing went wrong — the storm is one day out of
twelve, so it is by definition not typical, and a method built to find typical behaviour dropped
it. Whether that matters is the first question to ask yourself.

## 2  What are you optimising for?

Every lever below trades one kind of fidelity for another. There is no configuration that is best
at everything, so the choice starts with what your **downstream model** actually reads:

| If your result is driven by… | You care about | Look at |
|---|---|---|
| annual energy, utilisation, average cost | **totals** | the default; keep rescaling on |
| a peak that sizes capacity or a grid limit | **one specific period** | lever 3, extremes |
| storage sizing, peak pricing, duration curves | **the value distribution** | lever 2, `distribution` |
| unit commitment, ramping, intra-day storage | **shape within a day** | lever 4 — segment cautiously |
| anything with physically realistic profiles | **real days, not averages** | lever 2, `medoid` |
| seasonal storage across the calendar | **the sequence** | lever 1, and the assignments |

Keep your row in mind — the rest of this tutorial is the evidence behind it.

## 3  Lever 1 — how the days are grouped

This is the lever with the most options and, usually, the least leverage. Six methods, at the same
size:

In [ ]:
rows = []
for method in [
    "hierarchical",
    "kmeans",
    "kmedoids",
    "kmaxoids",
    "contiguous",
    "averaging",
]:
    r = tsam.aggregate(days, **COMMON, cluster=ClusterConfig(method=method))
    rows.append(
        {
            "method": method,
            "default representation": ClusterConfig(method=method).get_representation(),
            "weighted RMSE": round(float(r.accuracy.weighted_rmse), 4),
            "peak load kept": round(float(r.cluster_representatives["load"].max()), 2),
        }
    )
pd.DataFrame(rows).set_index("method")

Three things worth pulling out of that table.

**The `default representation` column is lever 2 in disguise.** Picking `kmeans` does not only
change how days are grouped — it silently swaps your representative from a real day to a synthetic
mean. Much of what looks like "a difference between clustering methods" is really the
representation tagging along. The two are independent, and the next section unbundles them.

**Two methods kept the peak, for completely different reasons.** `kmaxoids` kept it *by design*:
maximising spread means spending a centre on the outlier, and it paid for that in RMSE everywhere
else. `contiguous` kept it *by accident*: the storm sits between runs of unlike days, so the
adjacency constraint stranded it alone in a block of one — and a one-member cluster's representative
is that member. The first is a property you can rely on; the second is a coincidence of this
calendar that would vanish if the storm had a neighbour.

**The two calendar methods cost 3× the error of everything else.** `contiguous` and `averaging` are
not competing on fit and should not be judged on it — they buy a structural property instead.

For *why* each method carves the data the way it does, see
[Comparing clustering methods](comparing_clustering_methods.ipynb). The short version: start with
the default, and change this lever only when you need a property — real days (`kmedoids`), calendar
order (`contiguous`, `averaging`), or spread (`kmaxoids`).

## 4  Lever 2 — what each group becomes

Same grouping every time now — only the representation changes. This isolates the lever the table
above was hiding:

In [ ]:
from tsam.config import MinMaxMean

# The six representations. Most are plain strings; MinMaxMean needs to be told *which*
# column to take the max of — here, load, the one we care about preserving.
representations = {
    "medoid": "medoid",
    "mean": "mean",
    "maxoid": "maxoid",
    "distribution": "distribution",
    "distribution_minmax": "distribution_minmax",
    "minmax_mean (load=max)": MinMaxMean(max_columns=["load"], min_columns=[]),
}

rows = []
for name, rep in representations.items():
    r = tsam.aggregate(
        days, **COMMON, cluster=ClusterConfig(method="hierarchical", representation=rep)
    )
    rows.append(
        {
            "representation": name,
            "weighted RMSE": round(float(r.accuracy.weighted_rmse), 4),
            "duration-curve RMSE": round(float(r.accuracy.weighted_rmse_duration), 4),
            "peak load kept": round(float(r.cluster_representatives["load"].max()), 2),
        }
    )
pd.DataFrame(rows).set_index("representation")

**Same days, same groups, same size — and a real spread of outcomes.** Two error columns tell
different stories, and a representation can win one while losing the other:

* **weighted RMSE** asks *did the right thing happen at the right time?*
* **duration-curve RMSE** asks *did the right values occur as often as they should?* — timing
  ignored.

If your model sizes storage, prices peaks, or cares how many hours sit above a threshold, the
second column is your column, and `distribution` is built to win it. If your model cares when
things happen, the first is. Pick the representation for the error you actually care about — that
is this lever's whole job, and it is free.

## 5  Lever 3 — protecting a specific period

Levers 1 and 2 nudge *every* representative. Extremes do something different: they name **one
period** and force it in whole.

In [ ]:
kept = tsam.aggregate(
    days,
    **COMMON,
    cluster=ClusterConfig(method="hierarchical"),
    extremes=ExtremeConfig(method="new_cluster", max_value=["load"]),
)

print(f"typical periods:  {baseline.n_clusters} -> {kept.n_clusters}")
print(
    f"peak load kept:   {baseline.cluster_representatives['load'].max():.2f} -> "
    f"{kept.cluster_representatives['load'].max():.2f}"
)
print(f"cluster counts:   {dict(kept.cluster_counts)}")
print("\nThe storm now has a cluster to itself, with an occurrence count of 1 —")
print("it represents exactly one day, which is what a one-off event should do.")

## 6  Lever 4 — detail within a period

The other three levers all work on *which days* you keep. Segmentation works **inside** a day,
merging adjacent timesteps that look alike. Our designed days only have four timesteps, so there is
little to merge — but the mechanism is visible, and so is the cost:

In [ ]:
rows = []
for n_segments in [4, 3, 2]:
    r = tsam.aggregate(
        days,
        **COMMON,
        cluster=ClusterConfig(method="hierarchical"),
        segments=SegmentConfig(n_segments=n_segments),
    )
    rows.append(
        {
            "n_segments": n_segments,
            "timesteps in the model": r.n_clusters * n_segments,
            "weighted RMSE": round(float(r.accuracy.weighted_rmse), 4),
            "segment durations": str(r.segment_durations),
        }
    )
pd.DataFrame(rows).set_index("n_segments")

Note the durations are **not equal** — segments stretch where the profile is flat. That is the lever
working as intended, and also its catch: a downstream model must weight each segment by its
duration, and the segment boundaries differ from one typical period to the next.

Segmentation earns its place when your periods are long and repetitive — a 168-hour week has a lot
of flat overnight to reclaim; four six-hourly steps do not. **Cut periods first, segment second.**
See [How small can you go?](../how-to/tuning.ipynb) to search both levers at once.

## 7  Three ways to keep a peak — and they are not interchangeable

Here is the decision the docs are most often silent about. **Three different levers all claim to
preserve extremes**, and they are easy to confuse — the word "maxoid" alone appears in two of them:

* **`kmaxoids`** (lever 1) — the *grouping* spreads out to the edges of the data.
* **`maxoid`** (lever 2) — the *representation* picks each cluster's most extreme member.
* **`ExtremeConfig`** (lever 3) — a named period is *injected* whole.

They sound alike. They behave nothing alike. Same data, same k, all four side by side:

In [ ]:
approaches = {
    "baseline (hierarchical + medoid)": {
        "cluster": ClusterConfig(method="hierarchical")
    },
    "lever 1: kmaxoids grouping": {"cluster": ClusterConfig(method="kmaxoids")},
    "lever 2: maxoid representation": {
        "cluster": ClusterConfig(method="hierarchical", representation="maxoid")
    },
    "lever 3: ExtremeConfig injection": {
        "cluster": ClusterConfig(method="hierarchical"),
        "extremes": ExtremeConfig(method="new_cluster", max_value=["load"]),
    },
}

rows = []
for name, kwargs in approaches.items():
    r = tsam.aggregate(days, **COMMON, **kwargs)
    rows.append(
        {
            "approach": name,
            "typical periods": r.n_clusters,
            "peak load kept": round(float(r.cluster_representatives["load"].max()), 2),
            "weighted RMSE": round(float(r.accuracy.weighted_rmse), 4),
        }
    )
pd.DataFrame(rows).set_index("approach")

**All three recover the peak. They charge completely different prices for it.**

* **`kmaxoids`** keeps the peak and costs the *most* accuracy of the three. It spent one of only
  three centres on the outlier, so the remaining two must cover everything else — every ordinary
  day is reconstructed worse. You bought one peak with the whole dataset's fit.
* **`maxoid`** keeps the peak more cheaply, but it is a blunt instrument: it pushes **every**
  cluster's representative to that cluster's extreme, not just the one you cared about. The result
  is uniformly conservative — sometimes exactly what you want, often more than you asked for.
* **`ExtremeConfig`** keeps the peak **and has the best RMSE of all four** — but read the first
  column before celebrating: it is solving with **four** typical periods, not three. It is not
  beating the others at their own game; it is playing a different one, buying accuracy with size.

The rule that falls out:

> **If you can name the period that matters, name it** — `ExtremeConfig` is the only surgical
> option, and the only one whose cost you can see (one more period). Reach for `maxoid` when you
> cannot afford a period and want a generally conservative result. Reach for `kmaxoids` only when
> the **entire spread** matters, not one peak — that is the question it answers.

One trap worth flagging: `maxoid` and `kmaxoids` fight with rescaling. Extreme representatives
imply unrepresentative totals, so `preserve_column_means=True` may not fully converge and tsam will
warn. `ExtremeConfig` does not have this problem — extreme clusters are excluded from rescaling by
design. See [Rescaling](../explanation/how-aggregation-works/05_rescaling.ipynb).

## 8  Reading a configuration off your model

| You are modelling… | Start here |
|---|---|
| **energy balance, utilisation, average cost** | the default; leave `preserve_column_means=True` |
| **capacity sizing / reliability** | default + `ExtremeConfig(method="new_cluster", max_value=[…])` |
| **storage sizing, peak pricing** | `representation="distribution"` — the duration curve is the answer |
| **storage *and* firm peaks** | `representation="distribution_minmax"` |
| **unit commitment, ramping** | default; avoid segmentation, keep periods at full resolution |
| **seasonal storage** | default; use the ordered `cluster_assignments` (Kotzur et al. 2018) |
| **a contiguous-season structure** | `method="contiguous"` — accept the accuracy cost, it is a constraint |
| **something very large** | tune both levers: [How small can you go?](../how-to/tuning.ipynb) |

And the meta-rule, because most configurations are over-thought: **change one lever at a time, and
measure.** The default is strong. Levers 1 and 2 are free, so try them before spending periods on
lever 3 or detail on lever 4.

---

**Where to go next**

* [Comparing clustering methods](comparing_clustering_methods.ipynb) — why the six grouping methods
  disagree, worked through on these same 12 days
* [Comparing representations](comparing_representations.ipynb) — lever 2 the same way: all six on a
  single cluster, scored on whether each keeps a real day, the peak, or the mean
* [Representations](../how-to/representations.ipynb) — lever 2 on a realistic series
* [Extreme periods](../how-to/extreme_periods.ipynb) — lever 3, all four selection criteria
* [How small can you go?](../how-to/tuning.ipynb) — search levers 3 and 4 for a target size
* [How aggregation works](../explanation/how-aggregation-works/00_overview.ipynb) — the pipeline these levers
  hang off, traced by hand